In [3]:
import pandas as pd
import numpy as np

df_sqli = pd.read_csv('datasets/sqli.csv')
df_xss  = pd.read_csv('datasets/xss.csv')

print("=== sqli.csv ===")
print(f"Shape   : {df_sqli.shape}")
print(f"Columns : {df_sqli.columns.tolist()}")
print(f"Labels  : {df_sqli['Label'].value_counts().to_dict()}")
print(f"Nulls   : {df_sqli.isnull().sum().to_dict()}")
print()
print(df_sqli.head(3))

print()
print("=== xss.csv ===")
print(f"Shape   : {df_xss.shape}")
print(f"Columns : {df_xss.columns.tolist()}")
print(f"Labels  : {df_xss['Label'].value_counts().to_dict()}")
print(f"Nulls   : {df_xss.isnull().sum().to_dict()}")
print()
print(df_xss.head(3))

=== sqli.csv ===
Shape   : (30919, 2)
Columns : ['Query', 'Label']
Labels  : {0: 19537, 1: 11382}
Nulls   : {'Query': 0, 'Label': 0}

                                               Query  Label
0                  " or pg_sleep  (  __TIME__  )  --      1
1  create user name identified by pass123 tempora...      1
2   AND 1  =  utl_inaddr.get_host_address   (    ...      1

=== xss.csv ===
Shape   : (13686, 3)
Columns : ['Unnamed: 0', 'Sentence', 'Label']
Labels  : {1: 7373, 0: 6313}
Nulls   : {'Unnamed: 0': 0, 'Sentence': 0, 'Label': 0}

   Unnamed: 0                                           Sentence  Label
0           0  <li><a href="/wiki/File:Socrates.png" class="i...      0
1           1               <tt onmouseover="alert(1)">test</tt>      1
2           2  \t </span> <span class="reference-text">Steeri...      0


In [4]:
# SQLi: payload column is 'Query', label column is 'Label'
df_sqli = df_sqli.rename(columns={'Query': 'payload', 'Label': 'label'})
df_sqli = df_sqli[['payload', 'label']].copy()   # keep only these 2 columns
df_sqli['vuln_type'] = 'SQLi'

# XSS: payload column is 'Sentence', label is 'Label', drop 'Unnamed: 0'
df_xss = df_xss.rename(columns={'Sentence': 'payload', 'Label': 'label'})
df_xss = df_xss[['payload', 'label']].copy()     # drops 'Unnamed: 0' automatically
df_xss['vuln_type'] = 'XSS'

print("After standardising column names:")
print(f"SQLi columns: {df_sqli.columns.tolist()}")
print(f"XSS  columns: {df_xss.columns.tolist()}")
print()
print("Sample SQLi rows:")
print(df_sqli.head(3))
print()
print("Sample XSS rows:")
print(df_xss.head(3))

After standardising column names:
SQLi columns: ['payload', 'label', 'vuln_type']
XSS  columns: ['payload', 'label', 'vuln_type']

Sample SQLi rows:
                                             payload  label vuln_type
0                  " or pg_sleep  (  __TIME__  )  --      1      SQLi
1  create user name identified by pass123 tempora...      1      SQLi
2   AND 1  =  utl_inaddr.get_host_address   (    ...      1      SQLi

Sample XSS rows:
                                             payload  label vuln_type
0  <li><a href="/wiki/File:Socrates.png" class="i...      0       XSS
1               <tt onmouseover="alert(1)">test</tt>      1       XSS
2  \t </span> <span class="reference-text">Steeri...      0       XSS


In [5]:
sqli_before = len(df_sqli)
xss_before  = len(df_xss)

# Drop rows where payload is null
df_sqli = df_sqli.dropna(subset=['payload'])
df_xss  = df_xss.dropna(subset=['payload'])

# Drop rows where payload is only whitespace (important for XSS — 116 rows)
df_sqli = df_sqli[df_sqli['payload'].astype(str).str.strip() != '']
df_xss  = df_xss[df_xss['payload'].astype(str).str.strip() != '']

print("=== AFTER REMOVING NULLS AND EMPTY STRINGS ===")
print(f"SQLi: {sqli_before:,} → {len(df_sqli):,}  (removed {sqli_before - len(df_sqli)})")
print(f"XSS:  {xss_before:,} → {len(df_xss):,}  (removed {xss_before - len(df_xss)} empty rows)")

=== AFTER REMOVING NULLS AND EMPTY STRINGS ===
SQLi: 30,919 → 30,918  (removed 1)
XSS:  13,686 → 13,570  (removed 116 empty rows)


In [6]:
sqli_before = len(df_sqli)
xss_before  = len(df_xss)

df_sqli = df_sqli.drop_duplicates(subset=['payload', 'label'])
df_xss  = df_xss.drop_duplicates(subset=['payload', 'label'])

print("=== AFTER REMOVING DUPLICATES ===")
print(f"SQLi: {sqli_before:,} → {len(df_sqli):,}  (removed {sqli_before - len(df_sqli):,} duplicates)")
print(f"XSS:  {xss_before:,} → {len(df_xss):,}  (removed {xss_before - len(df_xss):,} duplicates)")
print()
print("Label distribution after all cleaning:")
print(f"  SQLi — 0 (benign): {(df_sqli['label']==0).sum():,}   1 (sqli): {(df_sqli['label']==1).sum():,}")
print(f"  XSS  — 0 (benign): {(df_xss['label']==0).sum():,}   1 (xss):  {(df_xss['label']==1).sum():,}")

=== AFTER REMOVING DUPLICATES ===
SQLi: 30,918 → 30,906  (removed 12 duplicates)
XSS:  13,570 → 10,914  (removed 2,656 duplicates)

Label distribution after all cleaning:
  SQLi — 0 (benign): 19,529   1 (sqli): 11,377
  XSS  — 0 (benign): 3,591   1 (xss):  7,323


In [7]:
print("=== LABEL ENCODING CHECK ===")
print(f"SQLi label dtype : {df_sqli['label'].dtype}")
print(f"XSS  label dtype : {df_xss['label'].dtype}")
print(f"SQLi unique labels: {sorted(df_sqli['label'].unique())}")
print(f"XSS  unique labels: {sorted(df_xss['label'].unique())}")
print()
print("Both datasets already use 0 (benign) and 1 (attack).")
print("No label encoding needed.")

# Ensure they are integers just in case
df_sqli['label'] = df_sqli['label'].astype(int)
df_xss['label']  = df_xss['label'].astype(int)
print("Labels confirmed as integers.")

=== LABEL ENCODING CHECK ===
SQLi label dtype : int64
XSS  label dtype : int64
SQLi unique labels: [np.int64(0), np.int64(1)]
XSS  unique labels: [np.int64(0), np.int64(1)]

Both datasets already use 0 (benign) and 1 (attack).
No label encoding needed.
Labels confirmed as integers.


In [8]:
df_sqli.to_csv('datasets/sqli_clean.csv', index=False)
df_xss.to_csv('datasets/xss_clean.csv',   index=False)

print("=== SAVED ===")
print(f"datasets/sqli_clean.csv  — {len(df_sqli):,} rows  columns: {df_sqli.columns.tolist()}")
print(f"datasets/xss_clean.csv   — {len(df_xss):,} rows  columns: {df_xss.columns.tolist()}")
print()
print("These are training data for Day 22.")
print()
print("Verify saved correctly:")
check_sqli = pd.read_csv('datasets/sqli_clean.csv')
check_xss  = pd.read_csv('datasets/xss_clean.csv')
print(f"sqli_clean.csv loaded: {check_sqli.shape}  — OK")
print(f"xss_clean.csv  loaded: {check_xss.shape}   — OK")

=== SAVED ===
datasets/sqli_clean.csv  — 30,906 rows  columns: ['payload', 'label', 'vuln_type']
datasets/xss_clean.csv   — 10,914 rows  columns: ['payload', 'label', 'vuln_type']

These are training data for Day 22.

Verify saved correctly:
sqli_clean.csv loaded: (30906, 3)  — OK
xss_clean.csv  loaded: (10914, 3)   — OK
